# CS610 Group 1: ensemble model in Colab

Combines the four trained models into one prediction of the hourly Citi Bike rentals per station.

| Model | Branch in the repo |
|---|---|
| Poisson regression | `feat/poisson-regression` |
| LightGBM | `feat/lightgbm` |
| Neural network | `feat/neural-network` |
| Random forest | `main` (`feat/random-forest`) |

The base models are **not retrained**. Each one has already saved:
- **out-of-fold predictions** on the 3 validation folds (Sep–Oct 2020, Nov–Dec 2020, Jan–Feb 2021), each made by a
  model trained only on the data before that fold
- **test predictions** (Mar–Apr 2021) from its final model

The ensemble learns how to combine the models from the out-of-fold predictions only. The test set is used once, at the
end, to report the final scores.

Runs on the default CPU runtime in about a minute. Run the cells from top to bottom.

## 1. Setup

In [ ]:
REPO_URL = 'https://github.com/amelia-lee-smu/CS610-Applied-Machine-Learning.git'

# the branch with each model's prediction files (data/predictions/<split>_<model>...parquet)
MODEL_BRANCHES = {
    'poisson_regression': 'feat/poisson-regression',
    'lightgbm': 'feat/lightgbm',
    'neural_network': 'feat/neural-network',
    'random_forest': 'main',
}

%cd /content
!rm -rf CS610-Applied-Machine-Learning
!git clone -q {REPO_URL}
%cd /content/CS610-Applied-Machine-Learning
!git log -1 --format='main: %h %ad %an: %s' --date=short

In [ ]:
# copy each model's prediction files from its branch into this folder
import subprocess

NUM_FOLDS = 3
for model_name, branch in MODEL_BRANCHES.items():
    files = [f'data/predictions/test_{model_name}.parquet'] + \
            [f'data/predictions/validation_{model_name}_fold{n}.parquet' for n in range(1, NUM_FOLDS + 1)]
    subprocess.run(['git', 'fetch', '-q', 'origin', branch], check = True)
    subprocess.run(['git', 'checkout', 'FETCH_HEAD', '--'] + files, check = True)
    commit = subprocess.run(['git', 'log', '-1', '--format=%h %ad %an', '--date=short', 'FETCH_HEAD'],
                            capture_output = True, text = True).stdout.strip()
    print(f'{model_name:20s} from {branch:25s} ({commit})')

!ls data/predictions

### Ensemble code

Written to `utilities/ensemble.py`, next to the team's model files. Four ways to combine the models are compared:

| Method | How it combines the predictions |
|---|---|
| `mean` | Simple average. Nothing is learned. |
| `weighted` | Weighted average. The weights are ≥ 0, add up to 1, and minimise the Poisson deviance. |
| `geometric` | Weighted average of the **log** predictions, then back to counts. Suits count models, whose errors are multiplicative. |
| `stacking` | A Poisson regression (the meta-model) on the log predictions of the models, with an intercept. |

In [ ]:
%%writefile utilities/ensemble.py
'''
Ensemble of the trained models.

Every base model has already made out-of-fold predictions on the 3 validation folds and predictions on the test set
(data/predictions/). The ensemble only learns how to combine those predictions, so the base models are not retrained.
'''
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance, root_mean_squared_error

from utilities.utilities import *
from utilities.values import *

KEY_COLUMNS = [STATION, DATE, HOUR_OF_DAY]
MIN_PREDICTION = 1e-6    # predictions are raised to this tiny value before taking logs or the Poisson deviance

ENSEMBLE_METHODS = {
    'mean': 'Simple average of the models',
    'weighted': 'Weighted average; the weights are learned and add up to 1',
    'geometric': 'Weighted geometric average (average of the log predictions); the weights add up to 1',
    'stacking': 'Poisson regression on the log predictions of the models (the weights do not need to add up to 1)',
}


def score(y_true, y_pred):
    '''MAE, RMSE and Poisson deviance, the same metrics as utilities/training.py.'''
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': root_mean_squared_error(y_true, y_pred),
        'poisson_deviance': mean_poisson_deviance(y_true, np.clip(y_pred, MIN_PREDICTION, None)),
    }


def read_predictions(file_paths):
    '''
    Reads one prediction file per model and puts them side by side: one column per model, one row per station-date-hour.
    file_paths: {model name: parquet file with a PREDICTION column}
    '''
    combined_df = None
    for model_name, file_path in file_paths.items():
        df = read_parquet_file(file_path)
        keep = KEY_COLUMNS + [NUM_RENTALS, PREDICTION] + (['fold'] if 'fold' in df.columns else [])
        df = df[keep].rename(columns = {PREDICTION: model_name})
        if combined_df is None:
            combined_df = df
        else:
            # every model must have predicted exactly the same rows, with the same actual rentals
            merged_df = combined_df.merge(df, on = [col for col in keep if col != PREDICTION], how = 'inner', validate = 'one_to_one')
            assert len(merged_df) == len(combined_df) == len(df), f'{model_name}: its rows do not match the other models'
            combined_df = merged_df
    return combined_df.reset_index(drop = True)


class EnsembleModel:
    '''
    Combines the predictions of several models into one prediction.
    method: one of ENSEMBLE_METHODS
    model_names: the prediction columns to combine
    '''
    def __init__(self, model_names, method = 'geometric'):
        assert method in ENSEMBLE_METHODS, f'method must be one of {list(ENSEMBLE_METHODS)}'
        self.model_names = list(model_names)
        self.method = method
        self.weights = np.full(len(self.model_names), 1 / len(self.model_names))
        self.stacker = None

    def _predictions(self, df):
        return np.clip(df[self.model_names].to_numpy(dtype = float), MIN_PREDICTION, None)

    def _combine(self, predictions, weights):
        if self.method == 'geometric':
            return np.exp(np.log(predictions) @ weights)
        return predictions @ weights

    def fit(self, df):
        '''df: out-of-fold predictions of the models (one column per model) and the actual rentals.'''
        predictions, y_true = self._predictions(df), df[NUM_RENTALS].to_numpy()

        if self.method in ('weighted', 'geometric'):
            # the weights are a softmax of free numbers, so they are always >= 0 and add up to 1
            to_weights = lambda z: np.exp(z - z.max()) / np.exp(z - z.max()).sum()
            loss = lambda z: mean_poisson_deviance(y_true, np.clip(self._combine(predictions, to_weights(z)), MIN_PREDICTION, None))
            result = minimize(loss, np.zeros(len(self.model_names)), method = 'Nelder-Mead',
                              options = {'maxiter': 4000, 'xatol': 1e-6, 'fatol': 1e-8})
            self.weights = to_weights(result.x)

        elif self.method == 'stacking':
            self.stacker = PoissonRegressor(alpha = 0, max_iter = 1000)
            self.stacker.fit(np.log(predictions), y_true)
            self.weights = self.stacker.coef_
        return self

    def predict(self, df):
        predictions = self._predictions(df)
        if self.method == 'stacking':
            return self.stacker.predict(np.log(predictions))
        return self._combine(predictions, self.weights)

    def weights_table(self):
        name = 'coefficient (log scale)' if self.method == 'stacking' else 'weight'
        return pd.Series(self.weights, index = self.model_names, name = name)

    def save_model(self, file_path):
        save_to_pickle_file(self, file_path)

    @staticmethod
    def load_model(file_path):
        return read_pickle_file(file_path)


def fold_date_ranges(df):
    '''First and last date of every validation fold.'''
    dates = pd.to_datetime(df[DATE])
    return dates.groupby(df['fold']).agg(['min', 'max'])


def check_no_time_leakage(df):
    '''
    The validation folds must be in chronological order and must not overlap: fold k ends before fold k+1 starts.
    Raises an AssertionError otherwise. Returns the date range of every fold.
    '''
    ranges = fold_date_ranges(df).sort_index()
    for earlier, later in zip(ranges.index[:-1], ranges.index[1:]):
        assert ranges.loc[earlier, 'max'] < ranges.loc[later, 'min'], f'fold {earlier} overlaps or comes after fold {later}'
    return ranges


def forward_chaining(df, model_names, method):
    '''
    Time-safe validation score of an ensemble method: learn the combination only on the folds BEFORE the held-out fold,
    then score it on the held-out fold (fold 2 <- fold 1; fold 3 <- folds 1 and 2).
    Fold 1 has no earlier fold to learn from, so it cannot be scored.
    Returns (one row of scores per scored fold, the out-of-fold ensemble predictions of those folds).
    '''
    check_no_time_leakage(df)
    rows, predictions = [], []
    folds = sorted(df['fold'].unique())
    for fold_number in folds[1:]:
        train_df, held_out_df = df[df['fold'] < fold_number], df[df['fold'] == fold_number]
        # the combination must never be learned from data that is later than the fold it is scored on
        assert pd.to_datetime(train_df[DATE]).max() < pd.to_datetime(held_out_df[DATE]).min(), 'time leakage'
        ensemble = EnsembleModel(model_names, method).fit(train_df)
        predicted = ensemble.predict(held_out_df)
        rows.append({'method': method, 'fold': fold_number, **score(held_out_df[NUM_RENTALS], predicted)})
        predictions.append(held_out_df[KEY_COLUMNS + [NUM_RENTALS, 'fold']].assign(**{PREDICTION: predicted}))
    return pd.DataFrame(rows), pd.concat(predictions, ignore_index = True)


def leave_one_fold_out(df, model_names, method):
    '''
    NOT time-safe - kept only to show how much the leak matters. Learns the combination on 2 of the 3 folds and scores
    the third, so fold 1 and fold 2 are scored with weights learned partly from LATER folds.
    Use forward_chaining() to choose the ensemble method.
    '''
    rows = []
    for fold_number in sorted(df['fold'].unique()):
        train_df, held_out_df = df[df['fold'] != fold_number], df[df['fold'] == fold_number]
        ensemble = EnsembleModel(model_names, method).fit(train_df)
        rows.append({'method': method, 'fold': fold_number, **score(held_out_df[NUM_RENTALS], ensemble.predict(held_out_df))})
    return pd.DataFrame(rows)

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from utilities.ensemble import *
from utilities.utilities import *
from utilities.values import *

data_dir = 'data/'
model_dir = 'models/'
MODELS = list(MODEL_BRANCHES)

# out-of-fold predictions of all models side by side (the 3 validation folds stacked)
validation_df = pd.concat([
    read_predictions({m: data_dir + f'predictions/validation_{m}_fold{n}.parquet' for m in MODELS})
    for n in range(1, NUM_FOLDS + 1)
], ignore_index = True)

# test predictions of all models side by side
test_df = read_predictions({m: data_dir + f'predictions/test_{m}.parquet' for m in MODELS})

print(f'validation: {len(validation_df):,} rows; test: {len(test_df):,} rows')
validation_df.head()

## 2. How good is each model, and how different are they?

An ensemble helps most when the models are good **and** make different mistakes.

In [ ]:
# each model's score on each validation fold (out-of-fold, so these are honest scores)
rows = []
for model_name in MODELS:
    for fold_number, fold_df in validation_df.groupby('fold'):
        rows.append({'model': model_name, 'fold': fold_number, **score(fold_df[NUM_RENTALS], fold_df[model_name])})
model_scores_df = pd.DataFrame(rows)

print('Poisson deviance per validation fold (lower is better):')
display(model_scores_df.pivot(index = 'model', columns = 'fold', values = 'poisson_deviance')
        .assign(mean = lambda d: d.mean(axis = 1)).sort_values('mean').round(4))

In [ ]:
# how alike the models' predictions are (1 = identical); lower values mean more to gain from combining them
correlation = validation_df[MODELS].corr(method = 'spearman')

fig, ax = plt.subplots(figsize = (6, 5))
image = ax.imshow(correlation, cmap = 'Blues', vmin = 0.5, vmax = 1)
ax.set_xticks(range(len(MODELS)), MODELS, rotation = 30, ha = 'right')
ax.set_yticks(range(len(MODELS)), MODELS)
for i in range(len(MODELS)):
    for j in range(len(MODELS)):
        ax.text(j, i, f'{correlation.iloc[i, j]:.2f}', ha = 'center', va = 'center',
                color = 'white' if correlation.iloc[i, j] > 0.85 else 'black')
fig.colorbar(image, ax = ax, label = 'Spearman correlation')
ax.set_title('Correlation of the validation predictions', loc = 'left')
plt.tight_layout()
plt.show()

## 3. Choose the ensemble method

The validation folds are in time order (Sep–Oct 2020, Nov–Dec 2020, Jan–Feb 2021), so an ensemble must only learn from
folds that come **before** the fold it is scored on (*forward chaining*):

| Scored fold | Weights learned on |
|---|---|
| fold 2 (Nov–Dec 2020) | fold 1 |
| fold 3 (Jan–Feb 2021) | folds 1 and 2 |

Fold 1 has nothing earlier to learn from, so it cannot be scored. The earlier leave-one-fold-out scored fold 1 and fold 2
with weights learned partly from **later** folds, which leaks the future into the validation score.
The method with the lowest mean Poisson deviance on folds 2 and 3 is chosen. The test set is not used here.

In [ ]:
print('Date range of each validation fold (must be in order and not overlap):')
display(check_no_time_leakage(validation_df))

results, FC_PREDS = [], {}
for method in ENSEMBLE_METHODS:
    scores, predictions = forward_chaining(validation_df, MODELS, method)
    results.append(scores); FC_PREDS[method] = predictions        # FC_PREDS: out-of-fold ensemble predictions, folds 2-3
cv_scores_df = pd.concat(results, ignore_index = True)
SCORED_FOLDS = [int(f) for f in sorted(cv_scores_df['fold'].unique())]

# the single models on the SAME folds, so the comparison is fair
same_folds_df = model_scores_df[model_scores_df['fold'].isin(SCORED_FOLDS)]
summary_df = pd.concat([
    cv_scores_df.groupby('method')[['poisson_deviance', 'MAE', 'RMSE']].mean().rename(index = lambda m: f'ensemble: {m}'),
    same_folds_df.groupby('model')[['poisson_deviance', 'MAE', 'RMSE']].mean(),
]).sort_values('poisson_deviance')

print(f'Mean over validation folds {SCORED_FOLDS} (lower is better):')
display(summary_df.round(4))

print('Poisson deviance of each ensemble method per scored fold:')
display(cv_scores_df.pivot(index = 'method', columns = 'fold', values = 'poisson_deviance').round(4))

BEST_METHOD = cv_scores_df.groupby('method')['poisson_deviance'].mean().idxmin()
print('Chosen method:', BEST_METHOD, '-', ENSEMBLE_METHODS[BEST_METHOD])

### How much did the leak matter?
Fold 2 is the one fold scored by both procedures. The old leave-one-fold-out learned its weights from folds 1 **and 3**
(fold 3 is in fold 2's future); forward chaining learns from fold 1 only. A negative difference means the old way looked better than it should have.

In [ ]:
old = pd.concat([leave_one_fold_out(validation_df, MODELS, m) for m in ENSEMBLE_METHODS], ignore_index = True)
fold2 = pd.DataFrame({
    'old (leaks fold 3)': old[old['fold'] == 2].set_index('method')['poisson_deviance'],
    'forward chaining':   cv_scores_df[cv_scores_df['fold'] == 2].set_index('method')['poisson_deviance'],
})
fold2['difference'] = fold2['forward chaining'] - fold2['old (leaks fold 3)']
print('Fold 2 Poisson deviance:')
fold2.round(4)

## 4. Final ensemble

The chosen method learns its weights on all 3 validation folds, then combines the models' test predictions.

In [ ]:
ensemble = EnsembleModel(MODELS, method = BEST_METHOD).fit(validation_df)
print(f'Method: {BEST_METHOD}')
display(ensemble.weights_table().round(3).to_frame())

# the weights learned by every method, for the report
weights_df = pd.concat({m: EnsembleModel(MODELS, m).fit(validation_df).weights_table().rename(None)
                        for m in ENSEMBLE_METHODS}, axis = 1)
print('Weights learned by each method (stacking: coefficients on the log scale):')
display(weights_df.round(3))

In [ ]:
# every method: learn on ALL 3 validation folds (all earlier than the test set), then predict the test set (Mar - Apr 2021)
FINAL_ENSEMBLES, TEST_PREDS = {}, {}
for method in ENSEMBLE_METHODS:
    fitted = EnsembleModel(MODELS, method).fit(validation_df)
    FINAL_ENSEMBLES[method], TEST_PREDS[method] = fitted, fitted.predict(test_df)
    fitted.save_model(model_dir + f'ensemble_{method}_model.pkl')
    save_to_parquet_file(test_df[KEY_COLUMNS + [NUM_RENTALS]].assign(**{PREDICTION: TEST_PREDS[method]}),
                         data_dir + f'predictions/test_ensemble_{method}.parquet')
    # out-of-fold (forward-chained) ensemble predictions on validation folds 2-3
    save_to_parquet_file(FC_PREDS[method], data_dir + f'predictions/validation_ensemble_{method}.parquet')

# the chosen method keeps the original file names
test_df[PREDICTION] = TEST_PREDS[BEST_METHOD]
ensemble = FINAL_ENSEMBLES[BEST_METHOD]
ensemble.save_model(model_dir + 'ensemble_model.pkl')
save_to_parquet_file(test_df, data_dir + 'predictions/test_ensemble.parquet')
print('All 4 ensemble methods saved (test_ensemble_<method>, validation_ensemble_<method>); chosen:', BEST_METHOD)
test_df.head()

## 5. Test set results

The final evaluation on Mar–Apr 2021. The test set was not used to choose the method or the weights, so these scores
show how the models do on data they have never seen. All ensemble methods are listed for the report, but the chosen
one was picked on the validation folds only.

In [ ]:
rows = [{'model': name, **score(test_df[NUM_RENTALS], test_df[name])} for name in MODELS]
for method in ENSEMBLE_METHODS:
    label = f'ensemble: {method}' + (' (chosen)' if method == BEST_METHOD else '')
    rows.append({'model': label, **score(test_df[NUM_RENTALS], TEST_PREDS[method])})

test_scores_df = pd.DataFrame(rows).set_index('model').sort_values('poisson_deviance')
print('Test set, Mar - Apr 2021 (lower is better):')
test_scores_df.round(4)

In [ ]:
# total rentals per day across all stations: actual vs the ensemble and the single models
daily_df = test_df.groupby(DATE)[[NUM_RENTALS, PREDICTION] + MODELS].sum()
daily_df.index = pd.to_datetime(daily_df.index)

fig, ax = plt.subplots(figsize = (12, 4.5))
ax.plot(daily_df.index, daily_df[NUM_RENTALS], color = 'black', linewidth = 2, label = 'actual')
ax.plot(daily_df.index, daily_df[PREDICTION], color = '#eb6834', linewidth = 2, label = f'ensemble ({BEST_METHOD})')
for model_name in MODELS:
    ax.plot(daily_df.index, daily_df[model_name], linewidth = 1, alpha = 0.45, label = model_name)
ax.set_ylabel('rentals per day (all stations)')
ax.set_title('Test period: daily rentals, actual vs predicted', loc = 'left')
ax.legend(ncol = 3, fontsize = 9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 6. All 8 models: choose on validation, then check on test

Eight models: the 4 base models and the 4 ensemble methods, plus two **historical baselines** for reference
(`baseline_last_week`: same station and hour 7 days earlier; `baseline_avg_4w`: mean of the same hour over the previous 4 weeks).

- **Validation** = folds 2–3 (the folds every ensemble method has an honest score on). The models are ranked here, and the choice is made here.
- **Test** is only shown afterwards, to see how well the choice generalises. Nothing is selected from it.
- Every score is shown on **all rows** and on **actual > 0 only**, because the many zero-demand hours make errors look small.
- The baselines forecast exactly 0 whenever there were no rentals in the reference hours, and Poisson deviance punishes a 0 forecast very hard when rentals do occur. Read the baselines mainly by MAE and RMSE.

In [ ]:
# ---- historical baselines: no training, only past rentals (needs data/processed in the repo)
try:
    history = pd.concat([read_parquet_file(data_dir + f'processed/{f}.parquet') for f in ('train_validation', 'test')])
    history = history[KEY_COLUMNS + [NUM_RENTALS]].drop_duplicates(KEY_COLUMNS)
    history['_ts'] = pd.to_datetime(history[DATE]) + pd.to_timedelta(history[HOUR_OF_DAY], unit = 'h')
except Exception as error:
    history = None
    print('Baselines skipped (could not read data/processed):', error)

BASELINES = ['baseline_last_week', 'baseline_avg_4w'] if history is not None else []

def add_baselines(df):
    out = df[KEY_COLUMNS].copy()
    out['_ts'] = pd.to_datetime(out[DATE]) + pd.to_timedelta(out[HOUR_OF_DAY], unit = 'h')
    lag_columns = []
    for weeks in (1, 2, 3, 4):
        lag = history[[STATION, '_ts', NUM_RENTALS]].copy()
        lag['_ts'] = lag['_ts'] + pd.Timedelta(weeks = weeks)       # line the history up with the hour being predicted
        out = out.merge(lag.rename(columns = {NUM_RENTALS: f'lag_{weeks}w'}), on = [STATION, '_ts'], how = 'left')
        lag_columns.append(f'lag_{weeks}w')
    out['baseline_last_week'] = out['lag_1w']
    out['baseline_avg_4w'] = out[lag_columns].mean(axis = 1)
    return out[['baseline_last_week', 'baseline_avg_4w']]

# ---- validation rows (folds 2-3) and test rows with one column per model
ENSEMBLE_COLUMNS = [f'ensemble_{m}' for m in ENSEMBLE_METHODS]
EIGHT_MODELS = MODELS + ENSEMBLE_COLUMNS

val_eval = validation_df[validation_df['fold'].isin(SCORED_FOLDS)].reset_index(drop = True)
for method in ENSEMBLE_METHODS:
    val_eval = val_eval.merge(FC_PREDS[method][KEY_COLUMNS + [PREDICTION]].rename(columns = {PREDICTION: f'ensemble_{method}'}),
                              on = KEY_COLUMNS, how = 'left', validate = 'one_to_one')
test_eval = test_df.copy()
for method in ENSEMBLE_METHODS:
    test_eval[f'ensemble_{method}'] = TEST_PREDS[method]
if BASELINES:
    val_eval[BASELINES] = add_baselines(val_eval).to_numpy()
    test_eval[BASELINES] = add_baselines(test_eval).to_numpy()
ALL_MODELS = EIGHT_MODELS + BASELINES

def table(df, columns, nonzero = False, by_fold = True):
    '''Mean over folds of MAE / RMSE / Poisson deviance. nonzero=True keeps only the hours with actual rentals > 0.'''
    d = df.dropna(subset = columns)
    d = d[d[NUM_RENTALS] > 0] if nonzero else d
    groups = [g for _, g in d.groupby('fold')] if by_fold else [d]
    return pd.DataFrame({c: pd.DataFrame([score(g[NUM_RENTALS], g[c]) for g in groups]).mean() for c in columns}).T

print(f'zero-rental share: validation (folds 2-3) {(val_eval[NUM_RENTALS] == 0).mean():.1%}   test {(test_eval[NUM_RENTALS] == 0).mean():.1%}')

### Validation: the ranking (folds 2-3, best first by Poisson deviance on all rows)

In [ ]:
val_table = pd.concat({'ALL rows': table(val_eval, ALL_MODELS),
                       'actual > 0 only': table(val_eval, ALL_MODELS, nonzero = True)}, axis = 1)
val_table = val_table.sort_values(('ALL rows', 'poisson_deviance'))
display(val_table.round(4))

best_all = val_table.loc[EIGHT_MODELS, ('ALL rows', 'poisson_deviance')].idxmin()
best_nz = val_table.loc[EIGHT_MODELS, ('actual > 0 only', 'poisson_deviance')].idxmin()
print(f'Best of the 8 on validation:  all rows -> {best_all}   |   actual > 0 only -> {best_nz}')

### Test: generalisation check only
Same order as the validation ranking. Compare each model's test score with its validation score; a large gap means it does not generalise.

In [ ]:
test_table = pd.concat({'ALL rows': table(test_eval, ALL_MODELS, by_fold = False),
                        'actual > 0 only': table(test_eval, ALL_MODELS, nonzero = True, by_fold = False)}, axis = 1).loc[val_table.index]
display(test_table.round(4))

gap = pd.DataFrame({
    'val (all)':  val_table[('ALL rows', 'poisson_deviance')],        'test (all)': test_table[('ALL rows', 'poisson_deviance')],
    'val (>0)':   val_table[('actual > 0 only', 'poisson_deviance')],  'test (>0)':  test_table[('actual > 0 only', 'poisson_deviance')],
})
gap['gap (all)'] = gap['test (all)'] - gap['val (all)']
gap['gap (>0)'] = gap['test (>0)'] - gap['val (>0)']
print('Poisson deviance, validation vs test:')
display(gap.round(4))

fig, axes = plt.subplots(1, 2, figsize = (14, 4.5))
for ax, (title, a, b) in zip(axes, [('all rows', 'val (all)', 'test (all)'), ('actual > 0 only', 'val (>0)', 'test (>0)')]):
    gap[[a, b]].rename(columns = {a: 'validation (folds 2-3)', b: 'test'}).plot.bar(ax = ax, rot = 60, title = f'Poisson deviance - {title}')
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

## 7. Save the results

Colab deletes its files when the session ends. This saves the ensemble code, the ensemble model, its test predictions
and the score tables to Google Drive (`MyDrive/CS610/ensemble/`) and downloads a zip.
Add them to the repo from your own computer (e.g. on a new `feat/ensemble` branch).

In [ ]:
summary_df.to_csv('ensemble_validation_scores.csv')
val_table.to_csv('model_comparison_validation.csv')
test_table.to_csv('model_comparison_test.csv')
test_scores_df.to_csv('ensemble_test_scores.csv')
weights_df.to_csv('ensemble_weights.csv')

outputs = [
    'utilities/ensemble.py',
    'models/ensemble_model.pkl',
    'data/predictions/test_ensemble.parquet',
    'ensemble_validation_scores.csv',
    'ensemble_test_scores.csv',
    'ensemble_weights.csv',
    'model_comparison_validation.csv',
    'model_comparison_test.csv',
]
for method in ENSEMBLE_METHODS:      # every ensemble method, not only the chosen one
    outputs += [f'models/ensemble_{method}_model.pkl',
                f'data/predictions/test_ensemble_{method}.parquet',
                f'data/predictions/validation_ensemble_{method}.parquet']

import shutil, zipfile
with zipfile.ZipFile('/content/ensemble_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for path in outputs:
        z.write(path)      # keeps the repo folder structure inside the zip

from google.colab import drive, files
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/CS610/ensemble'
os.makedirs(OUT, exist_ok = True)
for path in outputs:
    shutil.copy(path, OUT)
print('Saved to', OUT)

files.download('/content/ensemble_outputs.zip')